<a href="https://colab.research.google.com/github/elboudalicoding/Quality_Control_project/blob/main/NER_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
from datasets import load_dataset

dataset = load_dataset("wikiann", "ar")
#dataset

In [28]:
dataset["train"][2]

{'tokens': ['تحويل', 'نادي', 'باسوش', 'دي', 'فيريرا'],
 'ner_tags': [0, 3, 4, 4, 4],
 'langs': ['ar', 'ar', 'ar', 'ar', 'ar'],
 'spans': ['ORG: نادي باسوش دي فيريرا']}

**STEP 1 — Load AraBERT Tokenizer**

In [6]:
dataset["train"].features["ner_tags"]

List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']))

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

**Why this version?**

* It is trained on MSA

* It already includes proper Arabic normalization

* It is widely used in Arabic NER papers





In [8]:
label_list = dataset["train"].features["ner_tags"].feature.names

In [9]:
label_to_id = {l: i for i, l in enumerate(label_list)}

In [10]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            # Special tokens like [CLS], [SEP]
            labels.append(-100)

        elif word_idx != previous_word_idx:
            # First token of a word
            labels.append(example["ner_tags"][word_idx])

        else:
            # Subsequent subword token
            label_id = example["ner_tags"][word_idx]

            # If label is B-XXX, convert to I-XXX
            if label_id % 2 == 1:
                label_id += 1

            labels.append(label_id)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [11]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=False
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [12]:
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer
)

In [13]:
tokenizer = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv2")

In [14]:
num_labels = len(label_list)

model = AutoModelForTokenClassification.from_pretrained(
    "aubmindlab/bert-base-arabertv2",
    num_labels=num_labels
)

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you 

In [15]:
import transformers
print(transformers.__version__)

5.0.0


In [16]:
training_args = TrainingArguments(
    output_dir="./arabert-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    #logging_dir="./logs",
    load_best_model_at_end=True,
    report_to="none"   # prevents wandb error
)

In [17]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=9a8b6ee6463d7a84d6bda3d35ae98622370629deda07aece88f2bfd223713067
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [18]:
from seqeval.metrics import f1_score

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_predictions = []

    for pred, lab in zip(predictions, labels):
        temp_true = []
        temp_pred = []

        for p_i, l_i in zip(pred, lab):
            if l_i != -100:
                temp_true.append(label_list[l_i])
                temp_pred.append(label_list[p_i])

        true_labels.append(temp_true)
        true_predictions.append(temp_pred)

    return {"f1": f1_score(true_labels, true_predictions)}

In [19]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer)

In [20]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.139052,0.213383,0.876766
2,0.092944,0.226615,0.885475
3,0.066773,0.223848,0.892168


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3750, training_loss=0.09919857457478841, metrics={'train_runtime': 719.0218, 'train_samples_per_second': 83.447, 'train_steps_per_second': 5.215, 'total_flos': 869610418542624.0, 'train_loss': 0.09919857457478841, 'epoch': 3.0})

In [24]:
print(tokenized_dataset)

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})


In [25]:
trainer.evaluate(tokenized_dataset["test"])

{'eval_loss': 0.22904352843761444,
 'eval_f1': 0.8709069390259938,
 'eval_runtime': 17.6322,
 'eval_samples_per_second': 567.143,
 'eval_steps_per_second': 35.446,
 'epoch': 3.0}